In [5]:
import numpy as np
from scipy.integrate import dblquad, quad
from scipy.interpolate import CubicSpline

# ==========================================
# 1. CONSTANTS & CONVERSIONS (All to GeV)
# ==========================================
hbar_c = 0.197327  # GeV * fm

# Lead-208 Nuclear Parameters (in fm)
R_A_fm = 6.49
a_fm = 0.54
rho0_fm3 = 0.16  # Nuclear matter density

# Convert Nuclear Parameters to GeV units
R_A = R_A_fm / hbar_c  # ~ 33.55 GeV^-1
a = a_fm / hbar_c  # ~ 2.77 GeV^-1
rho0 = rho0_fm3 * (hbar_c**3)  # ~ 0.00123 GeV^3

# Total inelastic NN cross-section (e.g., 5.02 TeV LHC energy -> 70 mb)
sigma_NN_mb = 92.0
# 1 mb = 0.1 fm^2 -> Convert mb to fm^2, then to GeV^-2
sigma_NN = (sigma_NN_mb * 0.1) / (hbar_c**2)  # ~ 179.7 GeV^-2

# ==========================================
# STEP A: Pre-compute 1D Thickness TA(s)
# ==========================================
def woods_saxon(z, s):  #the first variable will be the integrated 
    r = np.sqrt(z**2 + s**2)
    return rho0 / (1.0 + np.exp((r - R_A) / a))


def compute_TA_scalar(s):
    # Integrate z from -3*R_A to 3*R_A (practically infinity for Woods-Saxon)
    z_limit = 3 * R_A
    result, _ = quad(woods_saxon, -z_limit, z_limit, args=(s,)) #we ignore the error estimate
    return result


# Generate 1D spline for TA(s)
s_grid = np.linspace(0, 3 * R_A, 200)
TA_values = np.array([compute_TA_scalar(s) for s in s_grid])
get_TA = CubicSpline(s_grid, TA_values, extrapolate=False)


# ==========================================
# STEP B & C: Compute Overlap Table & Spline
# ==========================================
def compute_TAA_scalar(b):
    # 2D Integrand using our fast 1D TA spline
    def integrand(phi, r):
        s1 = r
        s2 = np.sqrt(b**2 + r**2 - 2 * b * r * np.cos(phi))

        # Handle out-of-bounds for spline (where TA is essentially 0)
        val1 = get_TA(s1) if s1 < s_grid[-1] else 0.0
        val2 = get_TA(s2) if s2 < s_grid[-1] else 0.0

        return r * val1 * val2

    # Integrate r up to where nucleus ends
    r_max = 2.5 * R_A
    
    integral, _ = dblquad(integrand, 0, r_max, lambda r: 0, lambda r: np.pi)
    return 2.0 * integral  # Multiplied by 2 for angular symmetry of phi integral


print("Computing Gamma_AA(b) grid... (This may take a minute)")
# Generate a grid for total impact parameter b
# 0 to 150 GeV^-1 covers roughly 0 to 30 fm
b_grid = np.linspace(0, 150, 100)
print(b_grid)

TAA_values = np.array([compute_TAA_scalar(b) for b in b_grid])
Gamma_AA_values = np.exp(-sigma_NN * TAA_values)
   
np.savetxt(
    "Gamma_AA.dat",
    np.column_stack((b_grid, Gamma_AA_values)),
    header="b Gamma_AA"
)


Computing Gamma_AA(b) grid... (This may take a minute)
[  0.           1.51515152   3.03030303   4.54545455   6.06060606
   7.57575758   9.09090909  10.60606061  12.12121212  13.63636364
  15.15151515  16.66666667  18.18181818  19.6969697   21.21212121
  22.72727273  24.24242424  25.75757576  27.27272727  28.78787879
  30.3030303   31.81818182  33.33333333  34.84848485  36.36363636
  37.87878788  39.39393939  40.90909091  42.42424242  43.93939394
  45.45454545  46.96969697  48.48484848  50.          51.51515152
  53.03030303  54.54545455  56.06060606  57.57575758  59.09090909
  60.60606061  62.12121212  63.63636364  65.15151515  66.66666667
  68.18181818  69.6969697   71.21212121  72.72727273  74.24242424
  75.75757576  77.27272727  78.78787879  80.3030303   81.81818182
  83.33333333  84.84848485  86.36363636  87.87878788  89.39393939
  90.90909091  92.42424242  93.93939394  95.45454545  96.96969697
  98.48484848 100.         101.51515152 103.03030303 104.54545455
 106.06060606 107.575

In [3]:
# Create the final Step C Interpolator!
get_Gamma_AA = CubicSpline(b_grid, Gamma_AA_values, extrapolate=False)
print("Gamma_AA Spline successfully created!")

Gamma_AA Spline successfully created!


In [20]:
print(Gamma_AA_values[99])
print(Gamma_AA_values[-50:])

0.9999999999846346
[0.18134029 0.32947878 0.48874975 0.63240441 0.74722457 0.83174256
 0.89057539 0.93000518 0.9557561  0.97227529 0.98273975 0.98930867
 0.99340438 0.99594477 0.99751391 0.99847982 0.99907265 0.99943557
 0.99965722 0.9997923  0.99987446 0.99992432 0.99995452 0.99997276
 0.99998374 0.99999034 0.99999428 0.99999662 0.99999801 0.99999884
 0.99999932 0.9999996  0.99999977 0.99999987 0.99999992 0.99999995
 0.99999997 0.99999999 0.99999999 1.         1.         1.
 1.         1.         1.         1.         1.         1.
 1.         1.        ]


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.special import k1

# ==========================================
# CONSTANTS & SETUP
# ==========================================
alpha_em = 1.0 / 137.036
Z = 82.0              # Lead-208
m_N = 0.93827         # Nucleon mass (GeV)
hbarc = 0.197327      # GeV * fm conversion factor

# EMD Parameter for Pb-Pb at 5.36 TeV
# S is fm^2. Convert it to GeV^-2:
S_fm2 = 17.4*17.4
S_gev2 = S_fm2 / (hbarc**2)  

# ==========================================
# MULTI-CHANNEL LOG-SPACE INTEGRANDS
# ==========================================

def integrated_channels_wrapper(u, z, channel):
    """
    Computes the log-space integrand for different flux choices.
    channel options: 'PL(AnAn)', 'AnAn', 'An0n', 'Xn0n'
    """
    b = np.exp(u)
    zeta = z * m_N * b
    
    if zeta > 50.0:
        return 0.0
        
    # Core point-like photon density structure
    prefactor = (alpha_em * (Z**2)) / (np.pi**2)
    flux_density = (prefactor / z) * (zeta**2 / b**2) * (k1(zeta)**2)
    
    # 1. Determine Hadronic Survival Factor (Glauber)
    if channel == 'PL(AnAn)' and b >= (14.0 / hbarc):
        # Traditional Point-Like baseline uses a sharp cutoff at 2*R_A
        gamma = 1.0
         # IF they overlap there is no UPC
    elif channel == 'PL(AnAn)' and b < (14.0 / hbarc):
        gamma = 0.0
    else:
        # Realistic smooth Glauber for 'AnAn', 'An0n', and 'Xn0n'
        # IF the nucleus are very very far from each other, the probab to survive without 
        # having interaction is 1
        gamma = get_Gamma_AA(b) if b < 150.0 else 1.0
        
    # 2. Determine Electromagnetic Dissociation (EMD) Factors
    P_emd = S_gev2 / (b**2)
    P_no_emd = np.exp(-P_emd)
    
    if channel == 'An0n':
        # No breakup on the photon-emitting side, target can do anything
        emd_factor = P_no_emd
    elif channel == 'Xn0n':
        # No breakup on photon-emitting side AND guaranteed breakup on target side
        emd_factor = P_no_emd * (1.0 - P_no_emd)
    else:
        # PL(AnAn) and AnAn ignore single-side EMD neutron requirements
        emd_factor = 1.0
        
    # Return the unified integrand with log-space Jacobian exp(u)*exp(u)du
    return np.exp(2 * u) * flux_density * gamma * emd_factor


# ==========================================
# EXECUTION GRID LOOP
# ==========================================
z_grid = np.logspace(-4, -1, 100)

pl_flux_plot = []
anan_flux_plot = []
an0n_flux_plot = []
xn0n_flux_plot = []

# Lower limit set slightly above zero to handle the 1/b^2 profiles safely
b_min_numeric = 0.05 / hbarc  # ~0.05 fm

print("Computing all 4 UPC Flux Channels...")
for z in z_grid:
    b_max_dynamic = 60.0 / (z * m_N)
    print(b_max_dynamic)
    u_min = np.log(b_min_numeric)
    u_max = np.log(b_max_dynamic)
    
    # Compute Channel 1: PL Baseline
    int_pl, _ = quad(integrated_channels_wrapper, u_min, u_max, args=(z, 'PL(AnAn)'))
    pl_flux_plot.append(z * (2 * np.pi * int_pl))
    
    # Compute Channel 2: Total Geometric Flux (AnAn)
    int_anan, _ = quad(integrated_channels_wrapper, u_min, u_max, args=(z, 'AnAn'))
    anan_flux_plot.append(z * (2 * np.pi * int_anan))
    
    # Compute Channel 3: Neutron Tagged Flux (An0n)
    int_an0n, _ = quad(integrated_channels_wrapper, u_min, u_max, args=(z, 'An0n'))
    an0n_flux_plot.append(z * (2 * np.pi * int_an0n))
    
    # Compute Channel 4: Mutual Excitation Flux (Xn0n)
    int_xn0n, _ = quad(integrated_channels_wrapper, u_min, u_max, args=(z, 'Xn0n'))
    xn0n_flux_plot.append(z * (2 * np.pi * int_xn0n))

print("Calculation complete! Plotting all channels...")


Computing all 4 UPC Flux Channels...
639474.7780489624
596376.3182558665
556182.5504073135
518697.7079879057
483739.2184901759
451136.8141826113
420731.7028089366
392375.7941794905
365930.97888775426
341268.4556389725
318268.10391457315
296817.8989169073
276813.365944753
258157.07154208623
240758.1489417205
224531.85549245722
209399.15991416507
195286.35737048494
182124.71048434684
169850.1145478393
158402.78529581323
147726.9677224995
137770.6645229123
128485.3828363933
119825.89805879224
111750.03357291713
104218.45532441472
97194.4802425485
90643.89757277574
84534.80225091043
78837.43950731022
73524.05994422089
68568.78438042289
63947.47780489624
59637.631825586664
55618.25504073135
51869.770798790574
48373.9218490176
45113.681418261134
42073.17028089365
39237.57941794905
36593.097888775425
34126.84556389725
31826.810391457344
29681.789891690736
27681.3365944753
25815.707154208623
24075.814894172043
22453.18554924572
20939.915991416514
19528.635737048495
18212.471048434683
16985.011

In [ ]:
# ==========================================
# VISUALIZATION
# ==========================================
plt.figure(figsize=(8.5, 6.5))

# Plotting the 4 components to exactly match Figure 4
plt.plot(z_grid, anan_flux_plot, label="AnAn", color="red", linestyle="-", linewidth=2.2)
plt.plot(z_grid, an0n_flux_plot, label="An0n", color="blue", linestyle="--", linewidth=2.0)
plt.plot(z_grid, xn0n_flux_plot, label="Xn0n", color="green", linestyle="-.", linewidth=2.0)
plt.plot(z_grid, pl_flux_plot, label="PL(AnAn)", color="black", linestyle=":", linewidth=2.5)

plt.xscale('log')
plt.yscale('log')
plt.xlim(1e-4, 5e-1)
plt.ylim(1e-3, 300)

plt.xlabel(r"$z$", fontsize=13)
plt.ylabel(r"$z f_{\gamma/A}(z)$", fontsize=13)
plt.title("Pat Photon Flux at 5.36 TeV", fontsize=12, fontweight='bold')
#plt.grid(True, which="both", ls=":", alpha=0.5)
plt.legend(fontsize=11, loc="lower left")

plt.savefig("photon flux.png")
plt.show()